In [ ]:
# params to modify

queries = ['Driver: San Francisco', 'Driver: San Francisco review', 'Driver: San Francisco retrospective', 'Driver: San Francisco critique', 'Driver: San Francisco analysis']
cadence = {'years': 10}
join_q = False
refresh = False


# setup things

from googleapiclient.errors import HttpError
from googleapiclient.discovery import build
from concurrent.futures import ThreadPoolExecutor, as_completed

from tqdm.notebook import tqdm
from datetime import datetime
from itertools import product

import pandas as pd
import polars as pl

import logging

logging.basicConfig(level=logging.DEBUG)


if join_q and len(queries) > 1:
    join_queries = True
else:
    join_queries = False


start_date = (datetime.now() - pd.DateOffset(**cadence)).strftime("%Y-%m-%d")
end_date = datetime.now().strftime("%Y-%m-%d")


# cadence term
key, val = list(cadence.items())[0]
cadence_str =  str(key)[0] + str(val)


# create db anem
queries = sorted(queries)
joined_queries = '|'.join(queries)
split_joined_queries = joined_queries.split('|')
initials = ''.join([x[0] for x in split_joined_queries])
join_str = 'joined' if join_queries else 'split'
db_name = f'{initials.lower()}_{cadence_str}_{join_str}.db'
print(f'Database name: {db_name}')


# Get API keys
api_key_loc = '../../data/keys'
with open(api_key_loc, 'r') as f:
    API_KEYS = f.read().splitlines()

import duckdb

connection_name = f'../../data/{db_name}'
con = duckdb.connect(connection_name)

if refresh:
    print('Dropping tables...')
    con.execute(f'DROP TABLE IF EXISTS search_results')
    con.execute(f'DROP TABLE IF EXISTS channel_statistics')
    con.execute(f'DROP TABLE IF EXISTS channnel_playlists')
    con.execute(f'DROP TABLE IF EXISTS video_statistics')
    con.commit()
else:
    print('Tables not dropped...')
    # print all tables
    con.execute("SELECT name FROM sqlite_master WHERE type='table';")
    print(con.fetchall())
    
# con.close()

In [ ]:
# Parallel Search


tqdm.pandas()


keys = API_KEYS.copy()

def get_videos_for_period(query_string, start_date, pbar, cadence=None):

    logging.debug(f'Query: {query_string}, Start Date: {start_date}')

    next_page_token = None
    
    period = {key: 1 for key in cadence.keys()}    
    
    start_date = pd.Timestamp(start_date).strftime('%Y-%m-%dT00:00:00Z')
    end_date = (
        pd.Timestamp(start_date) + pd.DateOffset(**period)
        if cadence else datetime.now()
    ).strftime('%Y-%m-%dT00:00:00Z')       
    
    search_results_data = []
    while True:
        
        logging.debug(f'Querying with API key: {keys[0]}')
        
        youtube = build('youtube', 'v3', developerKey=keys[0])
        # First API request (search)
        request = youtube.search().list(
            part='snippet',
            maxResults=50,
            pageToken=next_page_token,
            publishedAfter=start_date,
            publishedBefore=end_date,
            q=query_string,
            relevanceLanguage='en',
            type='video',
            # videoCategoryId='20'
        )
        
        try:
            response = request.execute()
        except HttpError as e:
            
            if e.resp.status == 403:
                
                logging.error(f"API key {keys[0]} has exceeded its quota.")
                
                pbar.set_postfix_str(f"Keys remaining: {len(keys)}")
                
                keys.pop(0)
                
                continue
        
        next_page_token = response.get('nextPageToken')
        search_items = response.get('items')
        search_results_data.extend(search_items)
        
        if not next_page_token:
            break
        
    return search_results_data


# Function to parallelize the queries and date range processing
def parallel_search_query_execution(queries, period_start_dates, cadence):
    
    # Combine the queries and dates into a product list
    queries_and_dates = list(product(queries, period_start_dates))

    # Initialize progress bar and list for storing results
    pbar = tqdm(total=len(queries_and_dates), desc="Processing Queries and Dates")
    search_data = []

    # Use ThreadPoolExecutor to handle parallel processing
    with ThreadPoolExecutor(max_workers=4) as executor:  # Adjust max_workers as needed
        futures = []
        
        # Submit all query-date tasks for parallel execution
        for query, start_date in queries_and_dates:
            futures.append(executor.submit(get_videos_for_period, query, start_date, pbar, cadence))

        # Collect results as they are completed
        for future in as_completed(futures):
            video_data = future.result()
            search_data.append(video_data)
            pbar.update(1)

    search_results = [elem for sub in search_data for elem in sub]
    
    search_results_df = pl.json_normalize(search_results)

    return search_results_df


import logging


logging_level = logging.DEBUG

logging.basicConfig(level=logging_level)

logging.info(f'Logging level {logging_level}')


# Calculate number of periods
num_periods = list(cadence.values())[0] + 1
period_start_dates = pd.date_range(start=start_date, end=end_date, periods=num_periods)[:-1]

logging.info(f'Number of periods: {num_periods}')
logging.info(f'Initial start date: {start_date}')
logging.info(f'End date: {end_date}')

query_list = [joined_queries] if join_queries else queries

# Call the parallelized search execution function
search_results_df = parallel_search_query_execution(query_list, period_start_dates, cadence)

search_results_df.write_database(table_name='search_results',
                                 connection=f'duckdb:///{connection_name}',
                                 if_table_exists='replace')

search_results_df = pl.read_database(query=f'SELECT * FROM search_results', 
                         connection=con)

search_results_df.shape

In [ ]:
# Parallel get channel statistics


api_keys = API_KEYS.copy()


# Function to get channel statistics
def get_channel_statistics(channel_id):
    
    while True:
        
        youtube = build('youtube', 'v3', developerKey=api_keys[0])
        request = youtube.channels().list(
            part='contentDetails,id,snippet,statistics,topicDetails',
            id=channel_id
        )
        
        try:
            response = request.execute()
            items_data = response.get('items')
            return items_data
            
        except HttpError as e:
            if e.resp.status == 403:  # Quota exceeded
                logging.error(f"API key {api_keys[0]} has exceeded its quota.")
                api_keys.pop(0)
            elif e.resp.status in [400, 404, 503]:
                logging.error(f"Error fetching statistics for {channel_id}: {e}")
                return
            else:
                raise e

from concurrent.futures import ThreadPoolExecutor, as_completed

# Parallelized fetch
def parallel_channel_statistics_fetch(channel_id_strings):
    
    channels_data = []
    
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = [
            executor.submit(get_channel_statistics, channel_id_string)
            for channel_id_string in tqdm(channel_id_strings, desc="Fetching Channel Statistics")
        ]

        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing Results"):
            try:
                data = future.result()  # Get both data and the updated key index
                if data is not None:
                    channels_data.append(data)
            except Exception as e:
                print(f"Error processing a channel: {e}")
    
    flat_channel_data = [item for sublist in channels_data for item in sublist]
    
    channel_statistics_df = pl.json_normalize(flat_channel_data)
    
    return channel_statistics_df


import duckdb

connection_name = f'../../data/{db_name}'

con = duckdb.connect(connection_name)

logging.info(f"Found 'search_results' table. Getting channel statistics.")

channel_ids = con.sql('SELECT DISTINCT "snippet.channelId" FROM search_results').pl()

logging.info(f"Total unique channel IDs: {len(channel_ids)}")

batch_size = 50

channel_id_batches = (
    channel_ids
    .with_row_index()  # Add a row number to help with batch splitting
    .with_columns((pl.col("index") // batch_size).alias("batch"))  # Create batch numbers based on batch size
    .group_by("batch")  # Group by batch numbers
    .agg(
        pl.col("snippet.channelId").str.concat(delimiter=",").alias("joined_channelIds")  # Concatenate each batch into a single string
    )
)['joined_channelIds'].to_numpy()

channel_statistics_df = parallel_channel_statistics_fetch(channel_id_batches)


channel_statistics_df.write_database(table_name='channel_statistics',
                                 connection=f'duckdb:///{connection_name}',
                                 if_table_exists='replace')

channel_statistics_df = pl.read_database(query=f'SELECT * FROM channel_statistics', 
                         connection=con)

In [ ]:
# parallel get playlists


api_keys = API_KEYS.copy()


def get_channel_playlist_data(playlist_id, pbar):
    
    next_page_token = None
    channel_playlist_data = []
    
    while True:
        try:
            youtube = build('youtube', 'v3', developerKey=api_keys[0])

            request = youtube.playlistItems().list(
                part='contentDetails,snippet',
                playlistId=playlist_id,
                maxResults=50,
                pageToken=next_page_token
            )
            
            response = request.execute()
            response_data = response.get('items')
            channel_playlist_data.extend(response_data)

            pbar.update(len(response_data))

            next_page_token = response.get('nextPageToken')
            
            if not next_page_token:
                return channel_playlist_data
            
        except HttpError as e:
            if e.resp.status == 403:  # Quota exceeded
                logging.error(f"API key {api_keys[0]} has exceeded its quota.")
                api_keys.pop(0)

            elif e.resp.status == 404:
                print("Playlist not found.")
                break
            else:
                raise  # Re-raise the exception if it's not a quota error


connection_name = f'../../data/{db_name}'

con = duckdb.connect(connection_name)

table_names = con.execute("SELECT name FROM sqlite_master WHERE type='table';").pl()

# Main logic
if 'channel_statistics' in table_names['name']:

    logging.info(f"Found 'channel_statistics' table. Getting playlist info .")

    iteration = 0

    while True:

        table_names = con.execute("SELECT name FROM sqlite_master WHERE type='table';").pl()
        
        logging.info(f"Iteration {iteration}")

        channel_statistics_df = con.sql('SELECT * FROM channel_statistics').pl()

        # cast statistics.videoCount to int
        channel_statistics_df = channel_statistics_df.with_columns(
            pl.col('statistics.videoCount').cast(pl.Int32)
        )        

        logging.info(f"Total unique channel IDs: {channel_statistics_df.height}")
        
        if 'channel_playlists' in table_names['name']:
            
            logging.info(f"Found 'channel_playlists' table. Filtering channel IDs.")
            
            channel_playlists_ids = con.sql('SELECT DISTINCT "snippet.channelId" FROM channel_playlists').pl()
            
            channel_statistics_df = channel_statistics_df.filter(
                ~pl.col('id').is_in(channel_playlists_ids['snippet.channelId'])
                )
            
            logging.info(f"Channels to fetch: {channel_statistics_df.height}")
        
        number_of_videos = channel_statistics_df['statistics.videoCount'].sum()
        
        logging.info(f"Total number of videos: {number_of_videos}")
        
        if number_of_videos == 0:
            logging.info("All videos fetched.")
            break
    
        pbar = tqdm(total=number_of_videos, desc="Fetching Playlist Items")        

        channel_playlists_data = []
        
        def process_channel(channel_id):
            playlist_id = 'UU' + channel_id[2:]
            logging.info(f"Fetching videos for channel {channel_id} with playlist ID {playlist_id}")
            channel_playlist_data = get_channel_playlist_data(playlist_id, pbar)
            return channel_playlist_data
        
        with ThreadPoolExecutor(max_workers=4) as executor:
            futures = {executor.submit(process_channel, channel_id): channel_id for channel_id in channel_statistics_df['id']}

            for future in as_completed(futures):
                channel_id = futures[future]
                try:
                    channel_playlist_data = future.result()
                    if channel_playlist_data is not None:
                        channel_playlists_df = pl.json_normalize(channel_playlist_data)
                        channel_playlists_df.write_database(table_name='channel_playlists',
                                                            connection=f'duckdb:///{connection_name}',
                                                            if_table_exists='append')
                except Exception as e:
                    logging.error(f"Error processing channel {channel_id}: {e}")

        iteration += 1

In [ ]:
# Parallel video statistics fetch [suboptimal]


api_keys = API_KEYS.copy()


# Parameters
concurrent_requests = 64
batch_size = 1
wait_when_rate_limited = 60
timeout_seconds = 120
use_cookies = False
 
import aiohttp
import asyncio
from sqlalchemy.exc import ProgrammingError


semaphore = asyncio.Semaphore(concurrent_requests)


def fetch_video_statistics_for_id_string(id_string, pbar):
    
    logging.debug(f'Fetching video statistics for ID string: {id_string}')
        
    try:
        youtube = build('youtube', 'v3', developerKey=api_keys[0])

        request = youtube.videos().list(
            part='contentDetails,id,liveStreamingDetails,snippet,statistics,topicDetails',
            id=id_string
        )
        
        response = request.execute()
        
        items = response.get('items')
        
        logging.debug(f'Number of items: {len(items)}')
        
        pbar.update(len(items))
        
        return items
        
    except HttpError as e:
        
        if e.resp.status == 403:  # Quota exceeded
            logging.error(f"API key {api_keys[0]} has exceeded its quota.")
            api_keys.pop(0)
        else:
            raise  # Re-raise the exception if it's not a quota error


async def fetch_video_statistics_for_channel(channel, pbar):
    
    next_page_token = None 
    
    playlist_id = 'UU' + channel[2:]

    playlist_data = []

    async with semaphore:
        
        while True:
            
            try:
                youtube = build('youtube', 'v3', developerKey=api_keys[0])

                playlist_request = youtube.playlistItems().list(
                    part='contentDetails,snippet',
                    playlistId=playlist_id,
                    maxResults=50,
                    pageToken=next_page_token
                )
                
                playlist_response = playlist_request.execute()
                playlist_items = playlist_response.get('items')
                playlist_data.extend(playlist_items)
                
                next_page_token = playlist_response.get('nextPageToken')
                    
                if not next_page_token:
                    break
                    
            except HttpError as e:
                
                if e.resp.status == 403:  # Quota exceeded
                    logging.error(f"API key {api_keys[0]} has exceeded its quota.")
                    api_keys.pop(0)
                else:
                    raise  # Re-raise the exception if it's not a quota error
                
        video_ids = [item['snippet']['resourceId']['videoId'] for item in playlist_data]
        
        batch_size = 50
        video_id_batches = [video_ids[i:i + batch_size] for i in range(0, len(video_ids), batch_size)]
        video_id_strings = [','.join(video_id_batch) for video_id_batch in video_id_batches]
        
        video_statistics_data = [fetch_video_statistics_for_id_string(video_id_string, pbar) for video_id_string in video_id_strings]
        flattened_video_statistics_data = [item for sublist in video_statistics_data for item in sublist]
        
        result_df = pl.json_normalize(flattened_video_statistics_data)
        
        try:
            
            logging.debug(f"Writing video statistics to database for channel {channel}")
            
            result_df.write_database(table_name='video_statistics',
                                        connection=f'duckdb:///{connection_name}',
                                        if_table_exists='append')
        except ProgrammingError as e:
            if 'column' in str(e):
                existing_columns = con.execute("SELECT * FROM video_statistics LIMIT 1").fetch_df().columns
                for column in result_df.columns:
                    if column not in existing_columns:
                        try:
                            alter_stmt = f'ALTER TABLE video_statistics ADD COLUMN "{column}" TEXT'
                            con.execute(alter_stmt)
                            logging.info(f"Added missing column: {column}")
                        except Exception as e:
                            logging.error(f'Failed to add column: {column}')
                result_df.write_database(table_name='video_statistics',
                                        connection=f'duckdb:///{connection_name}',
                                        if_table_exists='append')        
        
        return 


async def fetch_data(channel_statistics_ids):
    
    pbar = tqdm(total=video_sums, desc="Fetching Video Statistics")
    
    tasks = [fetch_video_statistics_for_channel(channel_id, pbar) for channel_id in channel_statistics_ids]
    
    for task in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        await task

    pbar.close()
    

import duckdb

connection_name = f'../../data/{db_name}'
con = duckdb.connect(connection_name)

table_names = con.execute("SELECT name FROM sqlite_master WHERE type='table';").pl()

# Main logic
if 'channel_statistics' in table_names['name']:
    
    channel_statistics_df = con.sql('SELECT * FROM channel_statistics').pl()
    
    channel_statistics_df = channel_statistics_df.with_columns(
        pl.col('statistics.videoCount').cast(pl.Int32)
    )

    logging.info(f"Total unique channel IDs: {channel_statistics_df.shape[0]}")
    
    if 'video_statistics' in table_names['name']:
        
        video_statistics_ids = con.sql('SELECT DISTINCT "snippet.channelId" FROM video_statistics').pl()
        
        logging.info(f"Total unique video statistics channel IDs: {video_statistics_ids.shape[0]}")
        
        channel_statistics_df = channel_statistics_df.filter(
               ~pl.col('id').is_in(video_statistics_ids['snippet.channelId'])
            )

        
    logging.info(f"Channels to fetch: {channel_statistics_df.shape[0]}")
        
    video_sums = channel_statistics_df['statistics.videoCount'].sum()
    
    channel_statistics_ids = channel_statistics_df['id'].to_numpy().flatten()
        
    await fetch_data(channel_statistics_ids)

In [ ]:
import duckdb

connection_name = f'../../data/{db_name}'

con = duckdb.connect(connection_name)

video_statistics_df = con.execute(f'SELECT * FROM video_statistics').pl()

video_statistics_df

In [ ]:
def fetch_video_statistics_for_id_string(id_string):
    
    logging.debug(f'Fetching video statistics for ID string: {id_string}')
        
    try:
        youtube = build('youtube', 'v3', developerKey=api_keys[0])

        request = youtube.videos().list(
            part='contentDetails,id,liveStreamingDetails,snippet,statistics,topicDetails',
            id=id_string
        )
        
        response = request.execute()
        
        items = response.get('items')
        
        logging.debug(f'Number of items: {len(items)}')
        
        return items
        
    except HttpError as e:
        
        if e.resp.status == 403:  # Quota exceeded
            logging.error(f"API key {api_keys[0]} has exceeded its quota.")
            api_keys.pop(0)
        else:
            raise  # Re-raise the exception if it's not a quota error
        
        
api_keys = API_KEYS.copy()

# break into groups of 50
video_ids = search_results_df['id.videoId'].to_numpy().flatten()
batch_size = 50
video_id_batches = [video_ids[i:i + batch_size] for i in range(0, len(video_ids), batch_size)]

video_id_strings = [','.join(video_id_batch) for video_id_batch in video_id_batches]
video_statistics_data = [fetch_video_statistics_for_id_string(video_id_string) for video_id_string in tqdm(video_id_strings)]

no_none = [item for item in video_statistics_data if item]
flattened_video_statistics_data = [item for sublist in no_none for item in sublist]
result_df = pl.json_normalize(flattened_video_statistics_data)
result_df


In [ ]:
result_df.write_parquet(f'../../data/driver_san_francisco_video_statistics.parquet')

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import polars as pl
from tqdm.auto import tqdm
import logging

def get_channel_videos(channel_identifier, api_keys):
    """
    Get all videos from a YouTube channel.
    
    Parameters:
    -----------
    channel_identifier : str
        The channel ID or handle (with or without @ symbol)
    api_keys : list
        List of YouTube API keys to use
        
    Returns:
    --------
    polars.DataFrame
        DataFrame containing video title, ID, and description
    """
    api_keys_copy = api_keys.copy()
    
    # Step 1: Convert handle to channel ID if needed
    channel_id = None
    if channel_identifier.startswith('@') or not channel_identifier.startswith('UC'):
        # If this is a handle, we need to get the channel ID
        handle = channel_identifier
        if not handle.startswith('@'):
            handle = f'@{handle}'
            
        while True:
            try:
                youtube = build('youtube', 'v3', developerKey=api_keys_copy[0])
                request = youtube.search().list(
                    part='snippet',
                    q=handle,
                    type='channel',
                    maxResults=1
                )
                response = request.execute()
                
                if response.get('items'):
                    channel_id = response['items'][0]['snippet']['channelId']
                    logging.info(f"Converted handle {handle} to channel ID: {channel_id}")
                    break
                else:
                    raise ValueError(f"Could not find channel with handle: {handle}")
                    
            except HttpError as e:
                if e.resp.status == 403:  # Quota exceeded
                    logging.error(f"API key {api_keys_copy[0]} has exceeded its quota.")
                    api_keys_copy.pop(0)
                    if not api_keys_copy:
                        raise ValueError("All API keys have exceeded their quota.")
                else:
                    raise e
    else:
        channel_id = channel_identifier
    
    # Step 2: Create uploads playlist ID
    playlist_id = 'UU' + channel_id[2:]
    logging.info(f"Using uploads playlist ID: {playlist_id}")
    
    # Step 3: Get all videos from the uploads playlist
    next_page_token = None
    all_videos = []
    
    with tqdm(desc="Fetching videos") as pbar:
        while True:
            try:
                youtube = build('youtube', 'v3', developerKey=api_keys_copy[0])
                
                request = youtube.playlistItems().list(
                    part='contentDetails,snippet',
                    playlistId=playlist_id,
                    maxResults=50,
                    pageToken=next_page_token
                )
                
                response = request.execute()
                videos = response.get('items', [])
                all_videos.extend(videos)
                
                pbar.update(len(videos))
                pbar.set_postfix({"total": len(all_videos)})
                
                next_page_token = response.get('nextPageToken')
                if not next_page_token:
                    break
                    
            except HttpError as e:
                if e.resp.status == 403:  # Quota exceeded
                    logging.error(f"API key {api_keys_copy[0]} has exceeded its quota.")
                    api_keys_copy.pop(0)
                    if not api_keys_copy:
                        raise ValueError("All API keys have exceeded their quota.")
                elif e.resp.status == 404:
                    logging.error("Playlist not found.")
                    break
                else:
                    raise e
    
    # Step 4: Extract relevant data
    video_data = []
    for video in all_videos:
        video_data.append({
            'title': video['snippet']['title'],
            'video_id': video['snippet']['resourceId']['videoId'],
            'description': video['snippet']['description'],
            'published_at': video['snippet']['publishedAt'],
            'channel_id': video['snippet']['channelId'],
            'channel_title': video['snippet']['channelTitle']
        })
    
    # Create polars DataFrame
    if video_data:
        df = pl.DataFrame(video_data)
        logging.info(f"Found {len(df)} videos for channel {channel_id}")
        return df
    else:
        logging.warning(f"No videos found for channel {channel_id}")
        return pl.DataFrame({
            'title': [],
            'video_id': [],
            'description': [],
            'published_at': [],
            'channel_id': [],
            'channel_title': []
        })


INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
INFO:root:Converted handle @SecretBaseSBN to channel ID: UCGz-bVKK0HR2zZzJHEeYvWA
INFO:root:Using uploads playlist ID: UUGz-bVKK0HR2zZzJHEeYvWA
Fetching videos: 0it [00:00, ?it/s]INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
Fetching videos: 1it [00:00,  6.18it/s, total=1]
INFO:root:Found 1 videos for channel UCGz-bVKK0HR2zZzJHEeYvWA


Found 1 videos
shape: (1, 6)
┌──────────────────┬─────────────┬─────────────┬─────────────────┬─────────────────┬───────────────┐
│ title            ┆ video_id    ┆ description ┆ published_at    ┆ channel_id      ┆ channel_title │
│ ---              ┆ ---         ┆ ---         ┆ ---             ┆ ---             ┆ ---           │
│ str              ┆ str         ┆ str         ┆ str             ┆ str             ┆ str           │
╞══════════════════╪═════════════╪═════════════╪═════════════════╪═════════════════╪═══════════════╡
│ secret base～君  ┆ E9PXmSQu_5E ┆             ┆ 2024-07-02T12:5 ┆ UCGz-bVKK0HR2zZ ┆ ななさん      │
│ がくれたもの～歌 ┆             ┆             ┆ 8:39Z           ┆ zJHEeYvWA       ┆               │
│ の最高のギター指 ┆             ┆             ┆                 ┆                 ┆               │
│ 弾…              ┆             ┆             ┆                 ┆                 ┆               │
└──────────────────┴─────────────┴─────────────┴─────────────────┴─────────────────┴─

In [9]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import polars as pl
from tqdm.auto import tqdm
import logging
import re

def get_channel_videos(channel_identifier, api_keys):
    """
    Get all videos from a YouTube channel.
    
    Parameters:
    -----------
    channel_identifier : str
        The channel ID, username, or handle (with or without @ symbol)
    api_keys : list
        List of YouTube API keys to use
        
    Returns:
    --------
    polars.DataFrame
        DataFrame containing video title, ID, and description
    """
    api_keys_copy = api_keys.copy()
    
    # Step 1: Determine the channel ID
    channel_id = None
    
    # Check if it's already a channel ID (starts with UC)
    if channel_identifier.startswith('UC'):
        channel_id = channel_identifier
        logging.info(f"Using provided channel ID: {channel_id}")
    else:
        # Handle custom channel identifier (username or @handle)
        username = channel_identifier
        if username.startswith('@'):
            username = username[1:]  # Remove @ symbol
        
        while True and api_keys_copy:
            try:
                youtube = build('youtube', 'v3', developerKey=api_keys_copy[0])
                
                # Try with forUsername parameter (works for legacy usernames)
                try:
                    request = youtube.channels().list(
                        part='id',
                        forUsername=username
                    )
                    response = request.execute()
                    
                    if response.get('items'):
                        channel_id = response['items'][0]['id']
                        logging.info(f"Converted username {username} to channel ID: {channel_id}")
                        break
                except HttpError:
                    pass  # If this fails, it might be a newer @handle format
                
                # If we didn't get a channel ID, we need a different approach
                # For newer handles, we might need to use a different approach or custom URL
                if not channel_id:
                    logging.warning(f"Could not directly resolve username/handle: {username}")
                    logging.warning("Please provide a channel ID instead, or use a web browser to find the channel ID")
                    
                    # Ask the user for the channel ID or URL
                    user_input = input(f"Could not automatically resolve {channel_identifier}. Please enter the channel ID or URL manually: ")
                    
                    # Extract channel ID from URL if provided
                    if 'youtube.com' in user_input:
                        if 'channel/' in user_input:
                            channel_id = re.search(r'channel/([^/?]+)', user_input).group(1)
                        elif 'c/' in user_input:
                            # We need to do another lookup for custom URLs
                            custom_name = re.search(r'c/([^/?]+)', user_input).group(1)
                            logging.warning(f"Custom URLs (c/{custom_name}) require manual channel ID lookup")
                            channel_id = input("Please enter the channel ID directly (starts with UC): ")
                    else:
                        # Assume they entered the channel ID directly
                        channel_id = user_input
                    
                    if not channel_id.startswith('UC'):
                        raise ValueError("Invalid channel ID. Channel IDs should start with 'UC'")
                    break
                    
            except HttpError as e:
                if e.resp.status == 403:  # Quota exceeded
                    logging.error(f"API key {api_keys_copy[0]} has exceeded its quota.")
                    api_keys_copy.pop(0)
                    if not api_keys_copy:
                        raise ValueError("All API keys have exceeded their quota.")
                else:
                    raise e
    
    if not channel_id:
        raise ValueError(f"Could not determine channel ID for: {channel_identifier}")
    
    # Step 2: Create uploads playlist ID
    playlist_id = 'UU' + channel_id[2:]
    logging.info(f"Using uploads playlist ID: {playlist_id}")
    
    # Step 3: Get all videos from the uploads playlist
    next_page_token = None
    all_videos = []
    
    with tqdm(desc="Fetching videos") as pbar:
        while True and api_keys_copy:
            try:
                youtube = build('youtube', 'v3', developerKey=api_keys_copy[0])
                
                request = youtube.playlistItems().list(
                    part='contentDetails,snippet',
                    playlistId=playlist_id,
                    maxResults=50,
                    pageToken=next_page_token
                )
                
                response = request.execute()
                videos = response.get('items', [])
                all_videos.extend(videos)
                
                pbar.update(len(videos))
                pbar.set_postfix({"total": len(all_videos)})
                
                next_page_token = response.get('nextPageToken')
                if not next_page_token:
                    break
                    
            except HttpError as e:
                if e.resp.status == 403:  # Quota exceeded
                    logging.error(f"API key {api_keys_copy[0]} has exceeded its quota.")
                    api_keys_copy.pop(0)
                    if not api_keys_copy:
                        raise ValueError("All API keys have exceeded their quota.")
                elif e.resp.status == 404:
                    logging.error("Playlist not found.")
                    break
                else:
                    raise e
    
    # Step 4: Extract relevant data
    video_data = []
    for video in all_videos:
        video_data.append({
            'title': video['snippet']['title'],
            'video_id': video['snippet']['resourceId']['videoId'],
            'description': video['snippet']['description'],
            'published_at': video['snippet']['publishedAt'],
            'channel_id': video['snippet']['channelId'],
            'channel_title': video['snippet']['channelTitle']
        })
    
    # Create polars DataFrame
    if video_data:
        df = pl.DataFrame(video_data)
        logging.info(f"Found {len(df)} videos for channel {channel_id}")
        return df
    else:
        logging.warning(f"No videos found for channel {channel_id}")
        return pl.DataFrame({
            'title': [],
            'video_id': [],
            'description': [],
            'published_at': [],
            'channel_id': [],
            'channel_title': []
        })

In [10]:
# UCDRmGMSgrtZkOsh_NQl4_xw

# Example usage
if __name__ == "__main__":
    # Get API keys
    api_key_loc = '../../data/keys'
    with open(api_key_loc, 'r') as f:
        API_KEYS = f.read().splitlines()
    
    # Set up logging
    logging.basicConfig(level=logging.INFO)
    
    # Get videos for a channel
    channel_videos = get_channel_videos('@SecretBaseSBN', API_KEYS)
    
    # Display results
    print(f"Found {len(channel_videos)} videos")
    print(channel_videos.head())
    # Optionally save to file
    channel_videos.write_csv("channel_videos.csv")

INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
INFO:root:Using uploads playlist ID: UUDRmGMSgrtZkOsh_NQl4_xw
Fetching videos: 0it [00:00, ?it/s]INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
Fetching videos: 50it [00:00, 179.64it/s, total=50]INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
Fetching videos: 100it [00:00, 213.81it/s, total=100]INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
Fetching videos: 150it [00:00, 209.61it/s, total=150]INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
Fetching videos: 200it [00:00, 205.14it/s, total=200]INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
Fetching videos: 250it [00:01, 199.54it/s, total=250]INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
Fetching v

Found 3951 videos
shape: (5, 6)
┌─────────────────┬─────────────┬────────────────┬────────────────┬────────────────┬───────────────┐
│ title           ┆ video_id    ┆ description    ┆ published_at   ┆ channel_id     ┆ channel_title │
│ ---             ┆ ---         ┆ ---            ┆ ---            ┆ ---            ┆ ---           │
│ str             ┆ str         ┆ str            ┆ str            ┆ str            ┆ str           │
╞═════════════════╪═════════════╪════════════════╪════════════════╪════════════════╪═══════════════╡
│ The greatest    ┆ IGt2Gx-07O8 ┆ This might be  ┆ 2025-05-09T16: ┆ UCDRmGMSgrtZkO ┆ Secret Base   │
│ offensive       ┆             ┆ too many hits  ┆ 59:24Z         ┆ sh_NQl4_xw     ┆               │
│ perform…        ┆             ┆ an…            ┆                ┆                ┆               │
│ Shohei Ohtani   ┆ lmHkqblgdec ┆ In 2024, Los   ┆ 2025-05-09T16: ┆ UCDRmGMSgrtZkO ┆ Secret Base   │
│ crafted a       ┆             ┆ Angeles        ┆ 01:17Z  

In [11]:
channel_videos

title,video_id,description,published_at,channel_id,channel_title
str,str,str,str,str,str
"""The greatest offensive perform…","""IGt2Gx-07O8""","""This might be too many hits an…","""2025-05-09T16:59:24Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base"""
"""Shohei Ohtani crafted a season…","""lmHkqblgdec""","""In 2024, Los Angeles Dodgers s…","""2025-05-09T16:01:17Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base"""
"""Welcome to Year 2 of Top Secre…","""CrGp5O6FVv0""","""Year one flew by over at https…","""2025-05-05T17:00:14Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base"""
"""Kyrie Irving’s beef with Bosto…","""EMDCaFsqwrU""","""Starting in the 2017-18 season…","""2025-05-02T19:00:10Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base"""
"""Deion Sanders vs. Andre Rison …","""DGX9vRMYfms""","""Hopefully, Shedeur can learn f…","""2025-04-25T19:25:04Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base"""
…,…,…,…,…,…
"""Westminster Kennel Club Dog Sh…","""65rlXiWlbxU""","""Full story: http://www.sbnatio…","""2012-02-14T23:43:03Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base"""
"""A Super Bowl Gesture: Giants C…","""_p_rzRkz6YA""","""Gary Grimes nearly died in a c…","""2012-02-03T08:00:46Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base"""
"""Super Bowl Party Tips: Eli Kir…","""FGGggOPCok0""","""Top Chef's Eli Kirshtein helps…","""2012-02-02T18:23:26Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base"""


In [14]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import polars as pl
from tqdm.auto import tqdm
import logging
import re
import isodate  # For parsing ISO 8601 duration format

def get_channel_videos(channel_identifier, api_keys):
    """
    Get all videos from a YouTube channel.
    
    Parameters:
    -----------
    channel_identifier : str
        The channel ID (starts with UC)
    api_keys : list
        List of YouTube API keys to use
        
    Returns:
    --------
    polars.DataFrame
        DataFrame containing video title, ID, description, and duration
    """
    api_keys_copy = api_keys.copy()
    
    # Step 1: Determine the channel ID
    channel_id = None
    
    # Check if it's already a channel ID (starts with UC)
    if isinstance(channel_identifier, str) and channel_identifier.startswith('UC'):
        channel_id = channel_identifier
        logging.info(f"Using provided channel ID: {channel_id}")
    else:
        raise ValueError("Please provide a valid channel ID starting with 'UC'")
    
    # Step 2: Create uploads playlist ID
    playlist_id = 'UU' + channel_id[2:]
    logging.info(f"Using uploads playlist ID: {playlist_id}")
    
    # Step 3: Get all videos from the uploads playlist
    next_page_token = None
    all_video_ids = []
    
    with tqdm(desc="Fetching video IDs") as pbar:
        while True and api_keys_copy:
            try:
                youtube = build('youtube', 'v3', developerKey=api_keys_copy[0])
                
                request = youtube.playlistItems().list(
                    part='contentDetails',  # Only need contentDetails for videoId
                    playlistId=playlist_id,
                    maxResults=50,
                    pageToken=next_page_token
                )
                
                response = request.execute()
                videos = response.get('items', [])
                
                # Extract video IDs
                for video in videos:
                    all_video_ids.append(video['contentDetails']['videoId'])
                
                pbar.update(len(videos))
                pbar.set_postfix({"total": len(all_video_ids)})
                
                next_page_token = response.get('nextPageToken')
                if not next_page_token:
                    break
                    
            except HttpError as e:
                if e.resp.status == 403:  # Quota exceeded
                    logging.error(f"API key {api_keys_copy[0]} has exceeded its quota.")
                    api_keys_copy.pop(0)
                    if not api_keys_copy:
                        raise ValueError("All API keys have exceeded their quota.")
                elif e.resp.status == 404:
                    logging.error("Playlist not found.")
                    break
                else:
                    raise e
    
    # Step 4: Fetch detailed information for all videos in batches
    video_data = []
    batch_size = 50  # YouTube API allows up to 50 video IDs per request
    
    with tqdm(total=len(all_video_ids), desc="Fetching video details") as pbar:
        # Create batches of video IDs
        for i in range(0, len(all_video_ids), batch_size):
            batch_ids = all_video_ids[i:i+batch_size]
            id_string = ','.join(batch_ids)
            
            try:
                youtube = build('youtube', 'v3', developerKey=api_keys_copy[0])
                
                request = youtube.videos().list(
                    part='contentDetails,snippet,statistics',
                    id=id_string
                )
                
                response = request.execute()
                batch_videos = response.get('items', [])
                
                for video in batch_videos:
                    # Parse the ISO 8601 duration format
                    duration_iso = video['contentDetails']['duration']
                    duration_seconds = int(isodate.parse_duration(duration_iso).total_seconds())
                    
                    # Format duration as HH:MM:SS
                    hours, remainder = divmod(duration_seconds, 3600)
                    minutes, seconds = divmod(remainder, 60)
                    duration_formatted = f"{hours:02d}:{minutes:02d}:{seconds:02d}"
                    
                    video_data.append({
                        'title': video['snippet']['title'],
                        'video_id': video['id'],
                        'description': video['snippet']['description'],
                        'published_at': video['snippet']['publishedAt'],
                        'channel_id': video['snippet']['channelId'],
                        'channel_title': video['snippet']['channelTitle'],
                        'duration_iso': duration_iso,
                        'duration_seconds': duration_seconds,
                        'duration': duration_formatted,
                        'view_count': int(video['statistics'].get('viewCount', 0)),
                        'like_count': int(video['statistics'].get('likeCount', 0)),
                        'comment_count': int(video['statistics'].get('commentCount', 0))
                    })
                
                pbar.update(len(batch_ids))
                
            except HttpError as e:
                if e.resp.status == 403:  # Quota exceeded
                    logging.error(f"API key {api_keys_copy[0]} has exceeded its quota.")
                    api_keys_copy.pop(0)
                    if not api_keys_copy:
                        raise ValueError("All API keys have exceeded their quota.")
                else:
                    raise e
    
    # Create polars DataFrame
    if video_data:
        df = pl.DataFrame(video_data)
        logging.info(f"Found {len(df)} videos for channel {channel_id}")
        return df
    else:
        logging.warning(f"No videos found for channel {channel_id}")
        return pl.DataFrame({
            'title': [],
            'video_id': [],
            'description': [],
            'published_at': [],
            'channel_id': [],
            'channel_title': [],
            'duration_iso': [],
            'duration_seconds': [],
            'duration': [],
            'view_count': [],
            'like_count': [],
            'comment_count': []
        })

# Example usage
if __name__ == "__main__":
    # Get API keys
    api_key_loc = '../../data/keys'
    with open(api_key_loc, 'r') as f:
        API_KEYS = f.read().splitlines()
    
    # Set up logging
    logging.basicConfig(level=logging.INFO)

    # Get videos for a channel using the channel ID (must start with UC)
    channel_id = "UCDRmGMSgrtZkOsh_NQl4_xw"  # Replace with actual channel ID
    channel_videos = get_channel_videos(channel_id, API_KEYS)
    
    # Display results
    print(f"Found {len(channel_videos)} videos")
    print(channel_videos.select(['title', 'video_id', 'duration', 'view_count']).head())
    
    # Optionally save to file
    channel_videos.write_csv("channel_videos.csv")

INFO:root:Using provided channel ID: UCDRmGMSgrtZkOsh_NQl4_xw
INFO:root:Using uploads playlist ID: UUDRmGMSgrtZkOsh_NQl4_xw
Fetching video IDs: 0it [00:00, ?it/s]INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
Fetching video IDs: 50it [00:00, 223.08it/s, total=50]INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
Fetching video IDs: 100it [00:00, 249.60it/s, total=100]INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
Fetching video IDs: 150it [00:00, 246.91it/s, total=150]INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
Fetching video IDs: 200it [00:00, 237.99it/s, total=200]INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
Fetching video IDs: 250it [00:01, 254.70it/s, total=250]INFO:googleapiclient.discovery_cache:file_cache is only supported with oauth2client<4.0.0
Fetching video IDs: 

Found 3951 videos
shape: (5, 4)
┌─────────────────────────────────┬─────────────┬──────────┬────────────┐
│ title                           ┆ video_id    ┆ duration ┆ view_count │
│ ---                             ┆ ---         ┆ ---      ┆ ---        │
│ str                             ┆ str         ┆ str      ┆ i64        │
╞═════════════════════════════════╪═════════════╪══════════╪════════════╡
│ The greatest offensive perform… ┆ IGt2Gx-07O8 ┆ 00:02:47 ┆ 7475       │
│ Shohei Ohtani crafted a season… ┆ lmHkqblgdec ┆ 00:14:30 ┆ 93839      │
│ Welcome to Year 2 of Top Secre… ┆ CrGp5O6FVv0 ┆ 00:04:09 ┆ 19429      │
│ Kyrie Irving’s beef with Bosto… ┆ EMDCaFsqwrU ┆ 00:14:22 ┆ 217525     │
│ Deion Sanders vs. Andre Rison … ┆ DGX9vRMYfms ┆ 00:02:08 ┆ 10951      │
└─────────────────────────────────┴─────────────┴──────────┴────────────┘


In [13]:
!pip install isodate

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [15]:
channel_videos

title,video_id,description,published_at,channel_id,channel_title,duration_iso,duration_seconds,duration,view_count,like_count,comment_count
str,str,str,str,str,str,str,i64,str,i64,i64,i64
"""The greatest offensive perform…","""IGt2Gx-07O8""","""This might be too many hits an…","""2025-05-09T16:59:24Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT2M47S""",167,"""00:02:47""",7475,274,5
"""Shohei Ohtani crafted a season…","""lmHkqblgdec""","""In 2024, Los Angeles Dodgers s…","""2025-05-09T16:01:17Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT14M30S""",870,"""00:14:30""",93839,3178,280
"""Welcome to Year 2 of Top Secre…","""CrGp5O6FVv0""","""Year one flew by over at https…","""2025-05-05T17:00:14Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT4M9S""",249,"""00:04:09""",19429,701,120
"""Kyrie Irving’s beef with Bosto…","""EMDCaFsqwrU""","""Starting in the 2017-18 season…","""2025-05-02T19:00:10Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT14M22S""",862,"""00:14:22""",217525,6616,594
"""Deion Sanders vs. Andre Rison …","""DGX9vRMYfms""","""Hopefully, Shedeur can learn f…","""2025-04-25T19:25:04Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT2M8S""",128,"""00:02:08""",10951,368,4
…,…,…,…,…,…,…,…,…,…,…,…
"""Westminster Kennel Club Dog Sh…","""65rlXiWlbxU""","""Full story: http://www.sbnatio…","""2012-02-14T23:43:03Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT2M16S""",136,"""00:02:16""",10468,36,3
"""A Super Bowl Gesture: Giants C…","""_p_rzRkz6YA""","""Gary Grimes nearly died in a c…","""2012-02-03T08:00:46Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT4M3S""",243,"""00:04:03""",5737,31,3
"""Super Bowl Party Tips: Eli Kir…","""FGGggOPCok0""","""Top Chef's Eli Kirshtein helps…","""2012-02-02T18:23:26Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT4M39S""",279,"""00:04:39""",4729,21,6


In [29]:
channel_videos.filter(
    # pl.col('description').str.contains('Jon Bois'),
    pl.col('description').str.contains('Alex Rubenstein'),
    pl.col('description').str.contains('written') | pl.col('description').str.contains('Written'),
    pl.col('duration_seconds') >= 60 * 8,
    # pl.col('duration_seconds') <= 60 * 24,
).sort('duration')

title,video_id,description,published_at,channel_id,channel_title,duration_iso,duration_seconds,duration,view_count,like_count,comment_count
str,str,str,str,str,str,str,i64,str,i64,i64,i64
"""The worst NBA playoff game was…","""-8dGx-Ra8xA""","""When the Denver Nuggets visite…","""2019-02-21T16:00:44Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT8M5S""",485,"""00:08:05""",1611919,25534,1093
"""How many passes can you throw …","""mCW9n4TYkYs""","""A quarterback dropping back ti…","""2019-07-02T15:18:34Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT10M11S""",611,"""00:10:11""",329896,5453,327
"""Antonio Brown's beef with Ben …","""iVn37ndpkIc""","""For more Pennsylvania-based qu…","""2019-04-11T14:30:11Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT10M28S""",628,"""00:10:28""",2899759,32365,4229
"""How the Marlins accidentally w…","""NkOtndaStzo""","""The Florida Marlins became the…","""2019-03-27T17:00:12Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT12M9S""",729,"""00:12:09""",1423645,21956,2081
"""Correcting the NFL’s passer ra…","""5E7Z84WttIc""","""Want more for Secret Base? Goo…","""2024-09-17T17:00:03Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT14M""",840,"""00:14:00""",173598,9425,680
…,…,…,…,…,…,…,…,…,…,…,…
"""“You don’t belong here” | Dork…","""ymLqLKkRtQk""","""It’s the 1980s. The Vikings’ s…","""2023-08-23T01:00:09Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT1H22M55S""",4975,"""01:22:55""",548691,9192,1006
"""The two heroes | Dorktown""","""jGoZiKz6aqQ""","""The 1990s Minnesota Vikings we…","""2023-08-30T01:00:07Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT1H26M52S""",5212,"""01:26:52""",545034,8565,1195
"""It’s a funny story | The Histo…","""aV-8cG63o2w""","""Orange man bad! Football team …","""2021-09-07T17:00:16Z""","""UCDRmGMSgrtZkOsh_NQl4_xw""","""Secret Base""","""PT1H37M47S""",5867,"""01:37:47""",878759,19461,6049
